# Config

In [1]:
!git clone https://github.com/davidgc14/TalentCLEF-TaskA.git

Cloning into 'TalentCLEF-TaskA'...
remote: Enumerating objects: 675, done.
remote: Counting objects: 100% (461/461), done.
remote: Compressing objects: 100% (236/236), done.
remote: Total 675 (delta 269), reused 402 (delta 217), pack-reused 214 (from 1)
Receiving objects: 100% (675/675), 51.86 MiB | 27.80 MiB/s, done.
Resolving deltas: 100% (336/336), done.


In [2]:
!pip install ranx sentence-transformers

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.7/285.7 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.0/149.0 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 124.8 MB/s eta 0:00:00
  Created wheel for warc3-wet-clueweb09: filename=warc3_wet_clueweb09-0.2.5-py3-none-any.whl size=18919 sha256=e114eee6892b31ac697d7759bfb2e97bee57e51a11ab09186927da7c4427249c
  Stored in directory: /root/.cache/pip/wheels/f6/85/c2/9f0f621def52a1d5db7d29984f81e45f9fb6dfeb1a4eb6e31c
  Cr

In [15]:
import argparse
import itertools
import json
import os
import re
import time
import tempfile
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from ranx import Qrels, Run, evaluate
from sentence_transformers import SentenceTransformer, util, InputExample, losses, evaluation
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

In [4]:
today = time.strftime("%Y-%m-%d")

def set_directories():
  project_dir = Path(__name__).resolve().parents[1]
  project_dir = project_dir / 'content' / 'TalentCLEF-TaskA'
  date_dir = project_dir.parent / 'output' / today
  date_dir.mkdir(parents=True, exist_ok=True)

  # Crear subdirectorio incremental por ejecución (001, 002, ...)
  exec_dirs = sorted([d for d in date_dir.iterdir() if d.is_dir() and d.name.isdigit() and len(d.name) == 3])
  if exec_dirs and not any(exec_dirs[-1].iterdir()):
      output_dir = exec_dirs[-1]
  else:
      next_id = int(exec_dirs[-1].name) + 1 if exec_dirs else 1
      output_dir = date_dir / f"{next_id:03d}"
      output_dir.mkdir(exist_ok=True)

  return project_dir, output_dir

In [5]:
project_dir, output_dir = set_directories()
!cp ./TalentCLEF-TaskA/src/output/ranking_spanish_validation.csv {output_dir.parent.parent}
!rm -r sample_data/

## Evaluation file

In [6]:
def load_qrels(qrels_path):
    """
    Loads the qrels file (TREC format: q_id, iter, doc_id, rel)
    and converts it to a Qrels object.
    """
    qrels_df = pd.read_csv(qrels_path, sep="\t", header=None,
                           names=["q_id", "iter", "doc_id", "rel"],
                           dtype={"q_id": str, "doc_id": str, "rel":int})

    return Qrels.from_df(qrels_df, q_id_col="q_id", doc_id_col="doc_id", score_col="rel")

def load_run(run_path):
    """
    Loads the run file (TREC format: q_id, Q0, doc_id, rank, score, [tag])
    and converts it to a Run object.
    """
    run_df = pd.read_csv(run_path, sep=r"\s+", header=None)

    # Assign column names based on the number of columns
    if run_df.shape[1] == 5:
        run_df.columns = ["q_id", "Q0", "doc_id", "rank", "score"]
    elif run_df.shape[1] >= 6:
        run_df.columns = ["q_id", "Q0", "doc_id", "rank", "score", "tag"]
    else:
        raise ValueError("The run file does not have the expected format.")

    run_df["q_id"] = run_df.q_id.astype(str)
    run_df["doc_id"] = run_df.doc_id.astype(str)
    return Run.from_df(run_df, q_id_col="q_id", doc_id_col="doc_id", score_col="score")

def main():
    parser = argparse.ArgumentParser(
        description="Simplified evaluation script for TalentCLEF2025. "
                    "Requires qrels, run, query_lang, and corpuselements_lang as input."
    )
    parser.add_argument("--qrels", required=True, help="Path to the qrels file (TREC format)")
    parser.add_argument("--run", required=True, help="Path to the run file (TREC format)")
    args = parser.parse_args()

    # Display received parameters
    print("Received parameters:")
    print(f"  qrels: {args.qrels}")
    print(f"  run: {args.run}")

    print("Loading qrels...")
    qrels = load_qrels(args.qrels)
    print("Loading run...")
    run = load_run(args.run)

    # Define the evaluation metrics
    metrics = ["map", "mrr", "ndcg", "precision@5", "precision@10", "precision@100"]

    print("Running evaluation...")
    results = evaluate(qrels, run, metrics)

    print("\n=== Evaluation Results ===")
    for metric, score in results.items():
        print(f"{metric}: {score:.4f}")

def evaluate_run(qrels_path, run_path):
    """Evalúa un run y devuelve los resultados como dict."""
    qrels = load_qrels(qrels_path)
    run = load_run(run_path)
    metrics = ["map", "mrr", "ndcg", "precision@5", "precision@10", "precision@100"]
    return evaluate(qrels, run, metrics)


## Only spanish

In [7]:

# ========================
# DATA LOADING AND ENCODING
# ========================

def load_spanish_data(data_dir):
    """Load queries and corpus elements from Spanish data directory."""
    queries_path = data_dir / "queries"
    corpus_elements_path = data_dir / "corpus_elements"

    queries = pd.read_csv(queries_path, sep="\t")
    corpus_elements = pd.read_csv(corpus_elements_path, sep="\t")

    return (
        queries.q_id.to_list(),
        queries.jobtitle.to_list(),
        corpus_elements.c_id.to_list(),
        corpus_elements.jobtitle.to_list(),
    )


def encode_data(model, queries_texts, corpus_texts, model_name, device):
    """Encode queries and corpus using the model."""
    print('Encoding data:', model_name, 'on device:', device)
    query_embeddings = model.encode(queries_texts, convert_to_tensor=True, show_progress_bar=True, device=device)
    corpus_embeddings = model.encode(corpus_texts, convert_to_tensor=True, show_progress_bar=True, device=device)
    return query_embeddings, corpus_embeddings


def calculate_similarities_and_format_results(query_embeddings, corpus_embeddings, queries_ids, corpus_ids, model_name):
    """Calculate cosine similarities and format results in TREC format."""
    print('Calculating similarities and preparing results...')
    similarities = util.cos_sim(query_embeddings, corpus_embeddings).cpu().numpy()

    results = []
    for q_idx, q_id in enumerate(queries_ids):
        sorted_indices = np.argsort(-similarities[q_idx])  # Orden descendente
        for rank, c_idx in enumerate(sorted_indices):
            doc_id = corpus_ids[c_idx]
            score = similarities[q_idx, c_idx]
            results.append(f"{str(q_id)} Q0 {str(doc_id)} {rank+1} {score:.4f} {model_name}")
    return results


def run_evaluation_temp(qrels_path, results, model_name):
    """Run evaluation using temporary file without saving."""
    print('Evaluating Spanish monolingual performance...')

    # Crear archivo temporal
    with tempfile.NamedTemporaryFile(mode='w', suffix='.trec', delete=False, encoding='utf-8') as tmp_file:
        tmp_file.write("\n".join(results))
        tmp_path = tmp_file.name

    try:
        evaluation_results = evaluate_run(qrels_path, tmp_path)
    finally:
        # Eliminar archivo temporal
        os.unlink(tmp_path)

    return evaluation_results


def get_model_name(model):
    """Extract model name from the model object."""
    model_name = model[0].auto_model.config._name_or_path
    return model_name.split("/")[-1]


# ========================
# EVALUATION FUNCTION
# ========================

def spanish_monolingual_evaluation(model, device, source):
    """Evaluate model performance on Spanish monolingual data."""
    data_dir = project_dir / 'data' / source / 'spanish'
    qrels_path = data_dir / "qrels.tsv"

    print('Loading Spanish data...')
    queries_ids, queries_texts, corpus_ids, corpus_texts = load_spanish_data(data_dir)
    model_name = get_model_name(model)

    query_embeddings, corpus_embeddings = encode_data(model, queries_texts, corpus_texts, model_name, device)
    results = calculate_similarities_and_format_results(query_embeddings, corpus_embeddings, queries_ids, corpus_ids, model_name)

    evaluation_results = run_evaluation_temp(qrels_path, results, model_name)
    print('Spanish evaluation completed')
    return evaluation_results


# ========================
# SAVING RESULTS
# ========================

def save_spanish_results(evaluation_results, model_name, nickname, source):
    """Save Spanish monolingual evaluation results to JSON file."""
    print("Saving Spanish evaluation results...")

    json_path = output_dir / "results_spanish_monolingual.json"

    results_data = {
        "metadata": {
            "type": "spanish_monolingual",
            "model_name": model_name,
            "nickname": nickname,
            "source": source,
            "timestamp": today
        },
        "results": evaluation_results
    }

    with open(json_path, "w", encoding="utf-8") as jf:
        json.dump(results_data, jf, indent=2, ensure_ascii=False)

    print(f"Saved Spanish results to {json_path}")


# ========================
# RANKING
# ========================

def update_spanish_ranking(map_score, model_name, nickname, source):
    """Update Spanish-specific ranking CSV file with current execution results."""
    print("Updating Spanish ranking file...")

    ranking_file = output_dir.parent.parent / f"ranking_spanish_{source}.csv"
    execution_id = output_dir.name

    new_record = {
        'timestamp': today,
        'execution_id': execution_id,
        'model_name': model_name,
        'model_alias': nickname,
        'map_es_es': map_score
    }

    if ranking_file.exists():
        df = pd.read_csv(ranking_file)
    else:
        df = pd.DataFrame(columns=['timestamp', 'execution_id', 'model_name', 'model_alias', 'map_es_es'])

    # Double check for existing identical record
    comparison_cols = ['model_name', 'model_alias', 'map_es_es']

    if not df.empty:
        new_record_comparison = {k: new_record.get(k, np.nan) for k in comparison_cols}
        existing_records = df[comparison_cols].to_dict('records')

        for existing in existing_records:
            if all(abs(existing.get(k, np.nan) - new_record_comparison.get(k, np.nan)) < 1e-4
                   if isinstance(new_record_comparison.get(k), (float, int)) and not np.isnan(new_record_comparison.get(k, np.nan))
                   else existing.get(k) == new_record_comparison.get(k)
                   for k in comparison_cols):
                print("Ya existe un registro idéntico en el ranking español. No se agregará el nuevo registro.")
                return

    # Evitar warning de pandas con DataFrame vacío
    if df.empty:
        df = pd.DataFrame([new_record])
    else:
        df = pd.concat([df, pd.DataFrame([new_record])], ignore_index=True)

    df = df.sort_values('map_es_es', ascending=False).reset_index(drop=True)
    df.to_csv(ranking_file, index=False)

    print(f"Ranking español actualizado en {ranking_file}")


# ========================
# MAIN PIPELINE
# ========================

def run_spanish_evaluation(model_name, nickname, device, source):
    """Run complete Spanish monolingual evaluation pipeline."""
    model = SentenceTransformer(model_name, device=device)

    evaluation_results = spanish_monolingual_evaluation(model, device, source)
    save_spanish_results(evaluation_results, model_name, nickname, source)

    map_score = evaluation_results.get('map', np.nan)
    update_spanish_ranking(map_score, model_name, nickname, source)

    print(f"\n{'='*50}")
    print(f"Evaluación completada para {model_name}")
    print(f"MAP español-español: {map_score:.4f}")
    print(f"{'='*50}\n\n")


# ========================
# MAIN
# ========================

def main():
    parser = argparse.ArgumentParser(description='Evaluate sentence transformer models on Spanish monolingual job title matching')
    parser.add_argument('--model', type=str, default='paraphrase-multilingual-MiniLM-L12-v2',
                        help='Model name from HuggingFace or local path')
    parser.add_argument('--source', type=str, default='validation', choices=['validation', 'test'],
                        help='Dataset source to use for evaluation')
    parser.add_argument('--nickname', type=str, default='default', help='Alias for the model in results')
    parser.add_argument('--device', type=str, default='cpu', choices=['cpu', 'cuda'],
                        help='Device to run the model on')
    args = parser.parse_args()

    start_time = time.time()
    run_spanish_evaluation(args.model, args.nickname, args.device, args.source)
    end_time = time.time()

    print(f"Tiempo de ejecución: {round(end_time - start_time)} segundos.\n")

----------------------
# Training models
----------------------

In [8]:
device = 'cuda'
source = 'validation'

Modelos probados:
- paraphrase-multilingual-mpnet-base-v2
- paraphrase-multilingual-MiniLM-L12-v2
- BAAI/bge-m3

He visto que algunas propuestas interesantes involucran finetunning contrastivo o combinación de resultados de varios modelos sin entrenamiento. Se me ocurre que igual puedo plantear una mezcla de ambos enfoques, haciendo el finetunning unicamente en español y esperando que esto mejore los resultados.


# Basic evaluation

In [9]:
model_name = 'paraphrase-multilingual-mpnet-base-v2'
nickname = 'mpnet-base'

In [16]:
project_dir, output_dir = set_directories()
run_spanish_evaluation(model_name, nickname, device, source)

Loading Spanish data...
Encoding data: paraphrase-multilingual-mpnet-base-v2 on device: cuda


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Batches:   0%|          | 0/146 [00:00<?, ?it/s]

Calculating similarities and preparing results...
Evaluating Spanish monolingual performance...
Spanish evaluation completed
Saving Spanish evaluation results...
Saved Spanish results to /content/output/2025-11-26/002/results_spanish_monolingual.json
Updating Spanish ranking file...
Ya existe un registro idéntico en el ranking español. No se agregará el nuevo registro.

Evaluación completada para paraphrase-multilingual-mpnet-base-v2
MAP español-español: 0.4169




# Training

## Preparacion de los datos

In [10]:
training_dir = project_dir / 'data' / 'training' / 'spanish' / 'taskA_training_es.tsv'

train_df = pd.read_csv(training_dir, sep='\t', header=None)
train_df.columns = ['family_id', 'id', 'jobtitle_1', 'jobtitle_2']

train_df[['jobtitle_1', 'jobtitle_2']]

,jobtitle_1,jobtitle_2
0,jefe de escuadrón,instructor
1,comandante de aeronave,instructor de simulador
2,instructor,oficial del Ejército del Aire
3,comandante de aeronave,instructor
4,oficial de operaciones,instructora
...,...,...
20719,encargado de vestuarios,encargada de vestidores
20720,encargado de vestuarios/encargada de vestuarios,encargada de vestuarios
20721,acomodadora,acomodador/acomodadora
20722,acomodador,acomodador/acomodadora


In [11]:
# dividir las palabras con marcadores de genero, separados por /

expanded = []

for _, row in train_df.iterrows():
    jt1 = row['jobtitle_1']
    jt2 = row['jobtitle_2']

    # Obtener opciones para cada columna
    opts1 = [x.strip() for x in jt1.split('/')]
    opts2 = [x.strip() for x in jt2.split('/')]

    # Crear combinaciones A × B
    for a, b in itertools.product(opts1, opts2):
        if a != b:
            expanded.append({
                'jobtitle_1': a,
                'jobtitle_2': b
            })

df_expanded = pd.DataFrame(expanded)
df_expanded


,jobtitle_1,jobtitle_2
0,jefe de escuadrón,instructor
1,comandante de aeronave,instructor de simulador
2,instructor,oficial del Ejército del Aire
3,comandante de aeronave,instructor
4,oficial de operaciones,instructora
...,...,...
23868,encargado de vestuarios,encargada de vestidores
23869,encargado de vestuarios,encargada de vestuarios
23870,acomodadora,acomodador
23871,acomodador,acomodadora


In [12]:
# quedarnos solo con registros unicos, descartando los repetidos incluso cuando aparecen en la columna 1 y 2 intercambiados (ver ultimos tres registros del df_expanded como ejemplo)

df_temp = df_expanded.copy()

# Ordenar internamente cada pareja jobtitle_1 / jobtitle_2
df_temp[['jt1_sorted','jt2_sorted']] = (
    pd.DataFrame(
        df_temp.apply(lambda row: sorted([row['jobtitle_1'].strip(),
                                          row['jobtitle_2'].strip()]),
                      axis=1).to_list(),
        index=df_temp.index
    )
)

df_unique = df_temp.drop_duplicates(subset=['jt1_sorted','jt2_sorted'])
df_unique = df_unique[['jobtitle_1', 'jobtitle_2']].reset_index(drop=True)

df_unique

,jobtitle_1,jobtitle_2
0,jefe de escuadrón,instructor
1,comandante de aeronave,instructor de simulador
2,instructor,oficial del Ejército del Aire
3,comandante de aeronave,instructor
4,oficial de operaciones,instructora
...,...,...
17882,encargada de vestidores,encargada de vestuarios
17883,encargado de vestuarios,encargada de vestuarios
17884,encargado de vestuarios,encargado de vestidores
17885,encargada de vestidores,encargado de vestuarios


In [13]:
def normalize_titles(title):

    if pd.isna(title):
        return ""

    title = str(title).lower().strip()
    title = unicodedata.normalize('NFC', title)
    title = re.sub(r'[^\w\s]', '', title)   # creo que no esta funcionando correctamente. Revisar funciones de trabajos del master
    title = unicodedata.normalize('NFC', title)
    title = re.sub(r'\s+', ' ', title)
    title = title.strip('.,;:!?-')

    return title.strip()


def normalize_dataframe(df):

    df_normalized = df.copy()
    columns = df.columns.values

    for col in columns:
        if col in df_normalized.columns:
            df_normalized[col] = df_normalized[col].apply(normalize_titles)

    return df_normalized



df_normalized = normalize_dataframe(df_unique)
df_normalized

,jobtitle_1,jobtitle_2
0,jefe de escuadrón,instructor
1,comandante de aeronave,instructor de simulador
2,instructor,oficial del ejército del aire
3,comandante de aeronave,instructor
4,oficial de operaciones,instructora
...,...,...
17882,encargada de vestidores,encargada de vestuarios
17883,encargado de vestuarios,encargada de vestuarios
17884,encargado de vestuarios,encargado de vestidores
17885,encargada de vestidores,encargado de vestuarios


In [14]:
train_data = []

for idx, row in df_normalized.iterrows():
    sample = InputExample(texts=[row['jobtitle_1'], row['jobtitle_2']])
    train_data.append(sample)

print(f"\nCreados {len(train_data)} ejemplos de entrenamiento")


Creados 17887 ejemplos de entrenamiento


### Datos de validacion

In [36]:
validation_dir = project_dir / 'data' / 'validation' / 'spanish'

corpus_elements_val = pd.read_csv(validation_dir / 'corpus_elements', sep='\t')
queries_val = pd.read_csv(validation_dir / 'queries', sep='\t')

qrels_val = pd.read_csv(validation_dir / 'qrels.tsv', sep='\t', header=None)
qrels_val.columns = ['query_id', 'iter', 'doc_id', 'relevance']

In [38]:

# Convertir a diccionarios (ajustar nombres de columnas según tu dataset)
# Asumiendo que la primera columna es el ID y la segunda el jobtitle
queries = {}
for _, row in queries_val.iterrows():
    q_id = str(row.iloc[0])  # Primera columna como ID
    jobtitle = row.iloc[1]    # Segunda columna como texto
    queries[q_id] = jobtitle

corpus = {}
for _, row in corpus_elements_val.iterrows():
    c_id = str(row.iloc[0])  # Primera columna como ID
    jobtitle = row.iloc[1]    # Segunda columna como texto
    corpus[c_id] = jobtitle

# Crear relevant_docs desde qrels
relevant_docs = {}
for _, row in qrels_val.iterrows():
    q_id = str(row.iloc[0])  # Primera columna: q_id
    c_id = str(row.iloc[2])  # Tercera columna: c_id

    if q_id not in relevant_docs:
        relevant_docs[q_id] = set()
    relevant_docs[q_id].add(c_id)

print(f"\nValidación preparada:")
print(f"  - Queries: {len(queries)}")
print(f"  - Corpus: {len(corpus)}")
print(f"  - Pares relevantes: {sum(len(v) for v in relevant_docs.values())}")

# Crear evaluador
evaluator = evaluation.InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name='validation',
    show_progress_bar=False
)


Validación preparada:
  - Queries: 185
  - Corpus: 4661
  - Pares relevantes: 7579


## Entrenamiento del modelo

In [17]:
!pip uninstall wandb -y

Found existing installation: wandb 0.23.0
Uninstalling wandb-0.23.0:
  Successfully uninstalled wandb-0.23.0


In [24]:
model_name = 'paraphrase-multilingual-mpnet-base-v2'
nickname = 'mpnet-base'

model = SentenceTransformer(model_name, device=device)
train_loss = losses.MultipleNegativesRankingLoss(model)

In [41]:
model_output_dir = output_dir.parent.parent / 'finetune' / model_name #+ f'-({nickname})'
best_model_path = model_output_dir / 'best_model'
os.makedirs(model_output_dir, exist_ok=True)

In [39]:
epochs = 20
warmup_steps = 100
batch_size = 32
# evaluation_steps = 100
patience = 5
min_delta = 0.0001

train_dataloader = DataLoader(train_data, shuffle=True, batch_size=batch_size)

In [40]:
best_loss = float('inf')
best_epoch = 0
patience_counter = 0

print(f"\n{'='*60}")
print("INICIANDO ENTRENAMIENTO")
print(f"{'='*60}\n")

for epoch in range(epochs):
    print(f"\nÉpoca {epoch + 1}/{epochs}")
    print("-" * 40)

    # Entrenar una época
    model.fit(
        train_objectives=[(train_dataloader, train_loss)],
        epochs=1,
        warmup_steps=warmup_steps if epoch == 0 else 0,
        output_path=model_output_dir,
        show_progress_bar=True,
        use_amp=False,
        evaluator=evaluator,
        evaluation_steps=len(train_dataloader),
        save_best_model=False
    )

    # Calcular pérdida en entrenamiento
    model.eval()
    total_loss = 0
    num_batches = 0

    with torch.no_grad():
        for batch in train_dataloader:
            texts = []
            for sample in batch:
                texts.extend(sample.texts)

            features = model.tokenize(texts)
            features = {key: val.to(model.device) for key, val in features.items()}
            embeddings = model(features)['sentence_embedding']

            embeddings_a = embeddings[0::2]
            embeddings_b = embeddings[1::2]

            scores = torch.mm(embeddings_a, embeddings_b.t())
            labels = torch.arange(len(embeddings_a), device=model.device)

            loss_fn = torch.nn.CrossEntropyLoss()
            loss_value = loss_fn(scores, labels)

            total_loss += loss_value.item()
            num_batches += 1

    avg_train_loss = total_loss / num_batches
    model.train()

    # Evaluar métricas
    metrics = evaluator(model, output_path=model_output_dir)
    map_score = metrics.get('cosine_map@100', 0)

    print(f"\nResultados época {epoch + 1}:")
    print(f"  Loss:      {avg_train_loss:.4f}")
    print(f"  MAP@100:   {map_score:.4f}")

    # Early stopping
    if avg_train_loss < best_loss - min_delta:
        best_loss = avg_train_loss
        best_epoch = epoch + 1
        patience_counter = 0
        model.save(str(best_model_path))
        print(f"  ✓ Mejor modelo guardado (loss mejoró)")
    else:
        patience_counter += 1
        print(f"  ✗ Sin mejora ({patience_counter}/{patience})")

    if patience_counter >= patience:
        print(f"\n⚠ Early stopping en época {epoch + 1}")
        break


INICIANDO ENTRENAMIENTO


Época 1/20
----------------------------------------


Step,Training Loss,Validation Loss,Validation Cosine Accuracy@1,Validation Cosine Accuracy@3,Validation Cosine Accuracy@5,Validation Cosine Accuracy@10,Validation Cosine Precision@1,Validation Cosine Precision@3,Validation Cosine Precision@5,Validation Cosine Precision@10,Validation Cosine Recall@1,Validation Cosine Recall@3,Validation Cosine Recall@5,Validation Cosine Recall@10,Validation Cosine Ndcg@10,Validation Cosine Mrr@10,Validation Cosine Map@100
448,No log,No log,0.108108,0.989189,1.000000,1.000000,0.108108,0.596396,0.640000,0.637838,0.002813,0.085025,0.143694,0.248030,0.601701,0.548649,0.403163



Resultados época 1:
  Loss:      0.0766
  MAP@100:   0.0000


NameError: name 'best_model_path' is not defined

In [ ]:
print(f"\n{'='*60}")
print("ENTRENAMIENTO FINALIZADO")
print(f"{'='*60}")
print(f"Mejor época: {best_epoch}")
print(f"Mejor loss: {best_loss:.4f}")
print(f"Modelo guardado en: {best_model_path}")
print(f"{'='*60}")

In [27]:
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=epochs,
    warmup_steps=warmup_steps,
    output_path=model_output_dir,
    show_progress_bar=True,
    use_amp=True,
    evaluator=evaluator,
    evaluation_steps=evaluation_steps,
    save_best_model=True,
    checkpoint_path=model_output_dir,
    checkpoint_save_steps=evaluation_steps,
    checkpoint_save_total_limit=3,
)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss,Validation Cosine Accuracy@1,Validation Cosine Accuracy@3,Validation Cosine Accuracy@5,Validation Cosine Accuracy@10,Validation Cosine Precision@1,Validation Cosine Precision@3,Validation Cosine Precision@5,Validation Cosine Precision@10,Validation Cosine Recall@1,Validation Cosine Recall@3,Validation Cosine Recall@5,Validation Cosine Recall@10,Validation Cosine Ndcg@10,Validation Cosine Mrr@10,Validation Cosine Map@100
100,No log,No log,0.306596,0.616546,0.722471,0.805757,0.306596,0.205515,0.144494,0.080576,0.306596,0.616546,0.722471,0.805757,0.557906,0.477978,0.483763
200,No log,No log,0.318893,0.637507,0.743991,0.827557,0.318893,0.212502,0.148798,0.082756,0.318893,0.637507,0.743991,0.827557,0.576242,0.495032,0.501283
300,No log,No log,0.327278,0.653717,0.764952,0.853270,0.327278,0.217906,0.152990,0.085327,0.327278,0.653717,0.764952,0.853270,0.592401,0.508276,0.513586
400,No log,No log,0.332029,0.661263,0.778927,0.866126,0.332029,0.220421,0.155785,0.086613,0.332029,0.661263,0.778927,0.866126,0.601202,0.515672,0.521127
448,No log,No log,0.331750,0.673561,0.784516,0.874511,0.331750,0.224520,0.156903,0.087451,0.331750,0.673561,0.784516,0.874511,0.606296,0.519591,0.525088
500,0.281600,No log,0.330911,0.668250,0.783678,0.875629,0.330911,0.222750,0.156736,0.087563,0.330911,0.668250,0.783678,0.875629,0.605798,0.518698,0.524081
600,0.281600,No log,0.332588,0.678591,0.799609,0.882895,0.332588,0.226197,0.159922,0.088290,0.332588,0.678591,0.799609,0.882895,0.612054,0.524330,0.529268
700,0.281600,No log,0.335103,0.681386,0.799609,0.889044,0.335103,0.227129,0.159922,0.088904,0.335103,0.681386,0.799609,0.889044,0.614938,0.526358,0.531199
800,0.281600,No log,0.332029,0.689491,0.814142,0.899385,0.332029,0.229830,0.162828,0.089939,0.332029,0.689491,0.814142,0.899385,0.619276,0.528635,0.533068
896,0.281600,No log,0.336501,0.697317,0.816378,0.904695,0.336501,0.232439,0.163276,0.090470,0.336501,0.697317,0.816378,0.904695,0.625300,0.534813,0.539333


KeyboardInterrupt: 

In [ ]:
run_spanish_evaluation(str(model_output_dir), nickname + '-finetuned', device, source)

Loading Spanish data...
Encoding data: mpnet-base on device: cuda


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Batches:   0%|          | 0/146 [00:00<?, ?it/s]

Calculating similarities and preparing results...
Evaluating Spanish monolingual performance...
Spanish evaluation completed
Saving Spanish evaluation results...
Saved Spanish results to /content/output/2025-11-24/001/results_spanish_monolingual.json
Updating Spanish ranking file...
Ranking español actualizado en /content/output/ranking_spanish_validation.csv

Evaluación completada para /content/output/finetune/mpnet-base
MAP español-español: 0.4588


